# Gold v2 dev — 검증 구조 agent (설계·선택)

dev 세트(240장 / Q1·Q2 480문항, supervised train)에서 검증 agent의 구성을 고릅니다. 논문 수치는 확정된 구성을 gold v2에 한 번 돌려서 냅니다.

- **1단계**: dev 기준선의 `normal_ref_retrieved_diffdate` 답 재사용 (같은 날 촬영 사진을 뺀 비슷한 정상 이미지, 새 추론 없음)
- **검증 신호**: Q1·Q2 모순 / 원본만 Q1과 1단계 Q1이 다름
- **검증**: 결함 종류 문항을 새 근거로 한 번 더 묻고, 그 답으로 Q1·Q2를 다시 씀 (검증된 이미지는 모순 0)
  - `reask` 새 근거 없이 다시 묻기 (대조군) / `zoom` 가운데 확대 / `define` 결함 정의 문장 / `zoom_define` 둘 다 / `zoom_define_all` 모든 이미지
  - `rule_q1_first` 추가 호출 없이 Q1에 맞춰 Q2를 고치는 구조 규칙 (기준선)
- 보고: Q1 F1, Q2 F1, 모순, Joint, FDR, 문항당 호출 수, 답 번복(바꾼 수 / 맞게 바꾼 수 / 틀리게 바꾼 수), 1단계 대비 paired bootstrap CI

먼저 `lvlm_gold_v2_dev_baselines_colab.ipynb`(diffdate 조건 포함)를 끝내야 합니다.
GPU: **L4**. 호출 수는 라우팅된 이미지 수에 따라 다름(대략 모델당 500~700회, 1시간 안팎).
결과: `MyDrive/lvlm_insplad_results/gold_v2_verify_agent_dev/` → GitHub `results/latest/gold_v2_verify_agent_dev/`

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEV_BASELINES_FOLDER = "lvlm_insplad_results/gold_v2_dev_baselines"
RUN_FOLDER = "lvlm_insplad_results/gold_v2_verify_agent_dev"
MODELS = ["internvl3_5_8b","qwen3_5_4b"]
STAGE1_CONDITION = "normal_ref_retrieved_diffdate"
RULE = "diffdate"
VARIANTS = ["reask","zoom","define","zoom_define","zoom_define_all","rule_q1_first"]
TRIGGERS = ["contradiction","disagreement"]
CHECKPOINT_EVERY = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
RUN_ROOT = DRIVE_ROOT / RUN_FOLDER
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DEV_REFS_DIR = DEV_DIR / "refs"
STAGE1_DIR = DRIVE_ROOT / DEV_BASELINES_FOLDER / "predictions"

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0",
     "sentencepiece==0.2.2","pandas>=2.2","numpy>=1.26","pillow>=10","tqdm>=4.66","timm"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if not torch.cuda.is_available():
    raise RuntimeError("GPU Runtime(L4/A100)을 사용하세요.")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
DEV_QA = DEV_DIR / summary["file"]
REFS_CSV = DEV_REFS_DIR / "retrieved_refs.csv"
need = [DEV_QA, REFS_CSV] + [STAGE1_DIR / m / f"{STAGE1_CONDITION}.jsonl" for m in MODELS] \
       + [STAGE1_DIR / m / "image_only.jsonl" for m in MODELS]
missing = [str(p) for p in need if not p.is_file()]
if missing:
    raise FileNotFoundError("먼저 dev 기준선 노트북(diffdate 조건 포함)을 끝내세요. 없음: " + ", ".join(missing))
print("입력 확인 완료")

In [ ]:
# 실행 — 끊기면 이 셀만 다시 실행해도 완료된 호출은 재사용
run([sys.executable, REPO_DIR / "scripts/run_gold_v2_verify_agent.py",
     "--qa-file", DEV_QA, "--stage1-dir", STAGE1_DIR, "--stage1-condition", STAGE1_CONDITION,
     "--refs-csv", REFS_CSV, "--rule", RULE, "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP,
     "--out-dir", RUN_ROOT, "--models", *MODELS, "--variants", *VARIANTS, "--triggers", *TRIGGERS,
     "--checkpoint-every", str(CHECKPOINT_EVERY)])

In [ ]:
# 결과
pd.set_option("display.width", 220); pd.set_option("display.max_rows", 200)
print(json.dumps(json.loads((RUN_ROOT / "route_stats.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=1))
lb = pd.read_csv(RUN_ROOT / "leaderboard.csv")
display(lb[["model_key","variant","q1_anomaly_f1","q2_defect_macro_f1","consistency_error","joint_accuracy",
            "false_defect_rate","avg_lvlm_calls_per_q"]].round(3))
print("\n[답 번복] 바꾼 수 / 맞게 바꾼 수 / 틀리게 바꾼 수")
display(pd.read_csv(RUN_ROOT / "flips.csv"))
sig = pd.read_csv(RUN_ROOT / "significance.csv")
sig["txt"] = sig.apply(lambda r: f"{r.delta:+.3f} [{r.ci_low:+.3f},{r.ci_high:+.3f}]" + ("*" if r.significant else ""), axis=1)
print("\n[1단계 대비] * = CI가 0을 포함하지 않음 (모순·FDR은 낮을수록 좋음)")
display(sig.pivot_table(index=["model_key","condition"], columns="metric", values="txt", aggfunc="first"))
status = json.loads((RUN_ROOT / "run_status.json").read_text(encoding="utf-8"))
print("row_errors 합계:", sum(s["row_errors"] for s in status), "(0이 아니면 실행 셀을 다시 실행)")

In [ ]:
if PUBLISH:
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", RUN_ROOT,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_verify_agent_dev",
         "--files", "leaderboard.csv", "flips.csv", "significance.csv", "route_stats.json", "run_status.json",
         "run_metadata.json"], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_verify_agent_dev/")